# Nutosis

It is October 2018. The squirrels in Central Park are running into a problem and we need your help.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help).

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes the train or test set and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preparation should be noted here but actually implemented in the Preparation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit the train/test parquet files and a pickled preprocessor and final model along with these notebooks.

# Frame the Problem and Look at the Big Picture

1. Define the objective in business terms.
The wildlife conservation team needs a way to catch sick squirrels early even though they are short staffed. Our goal is to use volunteer sighting reports to flag squirrels that are likely sick, so staff can focus their limited time on the squirrels most worth capturing and testing instead of checking at random. Success means catching more sick squirrels early with the same number of staff hours and wasting fewer captures on healthy ones.

2. How will your solution be used?
Volunteers will keep submitting sightings through the app, which comes out to around 40 to 50 answered reports a day. Every few hours the model will score the newest batch of reports and produce a ranked list of squirrels that are likely sick, along with where and when they were seen. Staff can use that list to decide where to send people for capture and testing. Once test results come back, they get fed into the system as new samples so the model keeps improving over time.

3. What are the current solutions/workarounds (if any)?
The wildlife conservation team uses reported sightings and decide if its a credible source. If the team deems it as credible they go out, test the squirrels immune system and euthanize the squirrel at their lab if they show signs of Nutosis. 

4. How should you frame this problem (supervised/unsupervised, online/offline, …)?
This is a supervised learning problem because we have labeled outcomes from captures and testing that tell us whether a squirrel was sick. We would start with binary classification (sick or healthy) and could move to a multi class approach since Nutosis is evolving so (healthy, early stage, and advanced). Since new reports keep coming in, it fits a online batch setup where the model scores and retrains every few hours. 

5. How should performance be measured? Is the performance measure aligned with the business objective?
Our main metric will be **recall** because the client wants to miss as few sick sightings as possible. We will measure recall within the team's daily limit of 40-50 captures and use **precision** to see what percentage of the selected sightings are labeled as sick, since each trip uses limited staff time. We will also report F1 when comparing models.

6. What would be the minimum performance needed to reach the business objective?
The client did not specify a minimum recall target. We would consider the model useful if it identifies more sick sightings than the team's current behavior-based approach while staying within the same daily limit of 40-50 captures.

7. What are comparable problems? Can you reuse experience or tools?
This problem is similar to prioritizing emergency calls because the team has to decide which reports need attention first when staff are limited. We can use pandas to organize the sighting data, NumPy for calculations, Matplotlib to plot patterns in the reports, and scikit-learn to train and compare models.

8. Is human expertise available?
Since Nutosis is a new virus, the only way to diagnose a squirrel is by testing its immune system. The staff do not have the expertise to spot signs of Nutosis from a squirrel's appearance.

9. How would you solve the problem manually?
Staff would go through each volunteer report as it comes in, remove duplicate sightings of the same squirrel, and look for patterns like clusters of reports in the same area or unusual behavior. Based on that, they would pick which squirrels to capture, test their immune systems, and record whether each one was sick. Over time they would keep track of where positive cases showed up and use that to decide where to look next. 

10. List the assumptions you (or others) have made so far. Verify assumptions if possible.
We assume that volunteer reports contain enough detail to help us identify sick squirrels. We will check this by comparing the reported behaviors with the disease labels. The client confirmed that each row represents one sighting and that the same squirrel can be reported more than once. For this project, we treat sightings listed in `diseased.csv` as sick and all other sightings as healthy. All 379 IDs in the disease file matched the census data, giving us 379 sightings labeled as sick and 2,644 labeled as healthy.

# Get the Data

1. List the data you need and how much you need
    1. **Squirrel census:** We need all 3,023 Central Park sightings recorded from October 6–20, 2018.
    2. **Disease labels:** We need all 379 sighting IDs in `diseased.csv`. We will label matching sightings as sick and all other sightings as healthy.
    3. **Weather:** We need Central Park temperature and precipitation records for the census period. We will use only weather information available before each sighting.
    4. **Park events:** We need Central Park event permit dates and times to count how many events are scheduled during each census shift.

2. Find and document where you can get that data
    1. **Squirrel census:** We will use the 2018 Central Park squirrel sightings published by [NYC Open Data](https://data.cityofnewyork.us/Environment/2018-Central-Park-Squirrel-Census-Squirrel-Data/vfnx-vebw/about_data).
    2. **Disease labels:** Our client provided `diseased.csv`, which contains the sighting IDs labeled as sick.
    3. **Weather:** We will get temperature and precipitation records from NOAA’s [Central Park weather station](https://www.ncdc.noaa.gov/cdo-web/datasets/GHCND/stations/GHCND%3AUSW00094728/detail).
    4. **Park events:** We will get event locations and times from NYC Open Data’s [historical event permits](https://data.cityofnewyork.us/City-Government/NYC-Permitted-Event-Information-Historical/bkfu-528j/data).

3. Get access authorizations
The squirrel census, weather records, and event permits are publicly available through NYC Open Data and NOAA. Our client provided `diseased.csv` for use in this project.

4. Create a workspace (with enough storage space) <br>
We will use our local project folder to run the notebooks and cache downloaded data for reuse.

5. Get the data
We load the squirrel census and disease ID files from our project folder and download them from Kaggle if they are missing. We also download and cache Central Park weather records and event permits that overlap the census period.

6. Convert the data to a format you can easily manipulate (without changing the data itself)
We load the squirrel census, disease IDs, weather records, and event permits into pandas DataFrames. We read `diseased.csv` without a header so the first sighting ID is kept as data.

7. Ensure sensitive information is deleted or protected (e.g. anonymized)
We use sighting IDs to match reports with disease labels. We check the free-text notes for names or contact details and remove any we find before saving the training and test files. We keep the original data files out of the submission repository.

In [33]:
%pip install kagglehub scikit-learn matplotlib numpy ipywidgets pandas pyarrow
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import pandas as pd
import numpy as np
import shutil
from sklearn.model_selection import train_test_split

from pathlib import Path
import requests


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Functions

In [18]:
URL = "jamellalvarez/project2nutosisdataset"

def list_dataset_files():
    """Prints the files in the Kaggle dataset so you know which names to use."""
    cache_dir = kagglehub.dataset_download(URL)
    files = sorted(os.listdir(cache_dir))
    for f in files:
        print(f)
    return files

def get_local_file(filename):
    """
    Checks if filename exists locally. If not, fetches it from
    the Kaggle cache and copies it to your project directory.
    """
    if not os.path.exists(filename):
        print(f"Fetching {filename} from Kaggle...")
        cache_dir = kagglehub.dataset_download(URL)
        cached_file_path = os.path.join(cache_dir, filename)
        shutil.copy(cached_file_path, filename)
    return filename

def read_squirrel_data():
    """Loads the squirrel census sightings from the local directory."""
    return pd.read_csv(get_local_file("2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261104.csv"))


def read_sickness_data():
    """Loads the Nutosis test results from the local directory."""
    return pd.read_csv(get_local_file("diseased.csv"), header=None, names=["sighting_id"])

def download_csv(url, filename, params=None):
    """Download a CSV file if it is missing locally."""
    if not os.path.exists(filename):
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        with open(filename, "wb") as file:
            file.write(response.content)


# Data 

In [34]:
squirrel_data = read_squirrel_data()
sickness_data = read_sickness_data()

squirrel_data.shape
sickness_data.shape

squirrel_data.head()
sickness_data.head()


,sighting_id
0,11H-AM-1010-03
1,16I-AM-1008-01
2,12I-AM-1013-01
3,32F-PM-1008-07
4,33H-AM-1019-02


In [35]:
squirrel_data.info()
sickness_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3023 entries, 0 to 3022
Data columns (total 31 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   X                                           3023 non-null   float64
 1   Y                                           3023 non-null   float64
 2   Unique Squirrel ID                          3023 non-null   str    
 3   Hectare                                     3023 non-null   str    
 4   Shift                                       3023 non-null   str    
 5   Date                                        3023 non-null   int64  
 6   Hectare Squirrel Number                     3023 non-null   int64  
 7   Age                                         2902 non-null   str    
 8   Primary Fur Color                           2968 non-null   str    
 9   Highlight Fur Color                         1937 non-null   str    
 10  Combination of Primary 

### Central Park weather

The client noticed fewer reports on colder days. We collect daily minimum temperature, maximum temperature, and precipitation from the Central Park weather station. We use the previous day’s weather because the full daily summary would not be available when a morning sighting is reported. For sightings from October 6–20, we need weather from October 5–19.

In [21]:
weather_file = "central_park_weather_2018-10-05_to_2018-10-19.csv"
weather_url = "https://www.ncei.noaa.gov/access/services/data/v1"

weather_params = {
    "dataset": "daily-summaries",
    "stations": "USW00094728",
    "startDate": "2018-10-05",
    "endDate": "2018-10-19",
    "dataTypes": "TMIN,TMAX,PRCP",
    "units": "metric",
    "format": "csv",
}

download_csv(weather_url, weather_file, weather_params)
weather_data = pd.read_csv(weather_file, usecols=["DATE", "TMIN", "TMAX", "PRCP"], parse_dates=["DATE"])
weather_data.head(3)

,DATE,TMIN,TMAX,PRCP
0,2018-10-05,15.6,21.7,0.0
1,2018-10-06,17.2,20.0,0.0
2,2018-10-07,19.4,26.1,0.0


### Central Park events

The client told us that sightings depend on where and when people visit the park. We collect permits for Central Park events that overlap the census dates, including events that started before October 6. A permit tells us an event was scheduled. Attendance is unknown.

In [22]:
events_file = "central_park_events_2018-10-06_to_2018-10-20.csv"
events_url = "https://data.cityofnewyork.us/resource/bkfu-528j.csv"

event_params = {
    "$select": "event_id,event_name,event_location,start_date_time,end_date_time",
    "$where": (
        "event_borough = 'Manhattan' "
        "AND upper(event_location) like '%CENTRAL PARK:%' "
        "AND start_date_time < '2018-10-21T00:00:00' "
        "AND end_date_time >= '2018-10-06T00:00:00'"
    ),
    "$limit": 50000,
}

download_csv(events_url, events_file, event_params)
events_data = pd.read_csv(events_file, parse_dates=["start_date_time", "end_date_time"])
print(f"{len(events_data)} event records")
events_data[["event_name", "event_location", "start_date_time"]].head(3)

520 event records


,event_name,event_location,start_date_time
0,Central Park Squirrel Census,Central Park: Mineral Springs,2018-10-06 07:00:00
1,Grete's Great Gallop 10K,"Central Park: 72nd Street Cross Drive ,Central...",2018-10-06 08:00:00
2,Soccer - Non Regulation,"Central Park: North Meadow-Soccer-05 ,Central ...",2018-10-06 08:00:00


In [32]:
project_data = squirrel_data.copy()
disease_ids = sickness_data["sighting_id"]

project_data["diseased"] = project_data["Unique Squirrel ID"].isin(disease_ids)
matched_ids = disease_ids.isin(project_data["Unique Squirrel ID"]).sum()

print(f"{matched_ids} of {len(disease_ids)} disease IDs matched.")
print(project_data["diseased"].value_counts())

379 of 379 disease IDs matched.
diseased
False    2644
True      379
Name: count, dtype: int64


### Match sightings with the previous day’s weather

We match each sighting to the previous day’s weather because the full weather summary for the sighting day would not yet be available when the report came in.

In [36]:
sighting_dates = pd.to_datetime(project_data["Date"].astype(str), format="%m%d%Y")
previous_dates = sighting_dates - pd.Timedelta(days=1)
weather_by_date = weather_data.set_index("DATE")

project_data["prev_tmin_c"] = previous_dates.map(weather_by_date["TMIN"])
project_data["prev_tmax_c"] = previous_dates.map(weather_by_date["TMAX"])
project_data["prev_precip_mm"] = previous_dates.map(weather_by_date["PRCP"])

project_data[["Date", "prev_tmin_c", "prev_tmax_c", "prev_precip_mm"]].head(3)

,Date,prev_tmin_c,prev_tmax_c,prev_precip_mm
0,10142018,7.8,12.8,2.5
1,10192018,5.6,10.0,0.0
2,10142018,7.8,12.8,2.5


### Match sightings with scheduled park events

Every census shift overlaps at least one Central Park event permit, so we count distinct permits for each shift. We treat AM as midnight to noon and PM as noon to midnight. Each sighting receives the count for its date and shift. This count describes scheduled activity across the park.

In [37]:
shift_hours = project_data["Shift"].map({"AM": 0, "PM": 12})
shift_starts = sighting_dates + pd.to_timedelta(shift_hours, unit="h")

event_counts = {}
for start in shift_starts.unique():
    end = start + pd.Timedelta(hours=12)
    overlaps = (
        (events_data["start_date_time"] < end)
        & (events_data["end_date_time"] > start)
    )
    event_counts[start] = events_data.loc[overlaps, "event_id"].nunique()

project_data["park_event_count"] = shift_starts.map(event_counts)
project_data[["Date", "Shift", "park_event_count"]].drop_duplicates().head()

,Date,Shift,park_event_count
0,10142018,PM,27
1,10192018,AM,10
3,10172018,PM,32
4,10172018,AM,11
5,10102018,AM,10


In [38]:
added_columns = ["prev_tmin_c", "prev_tmax_c", "prev_precip_mm", "park_event_count"]

print("Total sightings:", len(project_data))
print("Sick sightings:", project_data["diseased"].sum())
print("Missing values per added column:")
print(project_data[added_columns].isna().sum())

Total sightings: 3023
Sick sightings: 379
Missing values per added column:
prev_tmin_c         0
prev_tmax_c         0
prev_precip_mm      0
park_event_count    0
dtype: int64


# Train and Test Split

The reason we went for this test/train split ...